In [1]:
import psycopg2
import psycopg2.extras

def get_near_routes(lon, lat, radius=1600, dbname="test", user="postgres", password="67500", host="localhost", port=5433):
    """
    Find routes within `radius` meters of a given point, returning each route's
    nearest actual GTFS shape point.

    Args:
        lon (float): Longitude
        lat (float): Latitude
        radius (int): Search radius in meters (default 1600 = ~20 min walking)
        dbname, user, password, host, port: Postgres connection params

    Returns:
        List of dicts: [{shape_id, dist_meters, closest_point}]
    """

    query = """
    WITH my_point AS (
      SELECT ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography AS geom
    ),
    shape_points AS (
      SELECT r.shape_id,
             ST_SetSRID(ST_MakePoint(r.shape_pt_lon, r.shape_pt_lat), 4326)::geography AS geom
      FROM gtfs_shapes_raw r
    ),
    ranked AS (
      SELECT sp.shape_id,
             sp.geom,
             ST_Distance(sp.geom, p.geom) AS dist_meters,
             ROW_NUMBER() OVER (PARTITION BY sp.shape_id ORDER BY sp.geom <-> p.geom) AS rn
      FROM shape_points sp, my_point p
      WHERE ST_DWithin(sp.geom, p.geom, %s)
    )
    SELECT shape_id,
           ST_AsGeoJSON(sp.geom::geometry) AS closest_point,
           dist_meters
    FROM ranked sp
    WHERE rn = 1
    ORDER BY dist_meters;
    """

    conn = psycopg2.connect(
        dbname=dbname, user=user, password=password, host=host, port=port
    )
    cur = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)

    cur.execute(query, (lon, lat, radius))
    rows = cur.fetchall()

    cur.close()
    conn.close()

    return [
        {
            "shape_id": row["shape_id"],
            "closest_point": row["closest_point"],  # GeoJSON string
            "dist_meters": float(row["dist_meters"])
        }
        for row in rows
    ]


In [1]:
import psycopg2
import psycopg2.extras

def get_near_routes(lon, lat, radius=1600,
                    dbname="osm", user="postgres",
                    password="67500", host="localhost", port=5433):
    """
    Find GTFS routes within `radius` meters of a given point.
    Uses gtfs_shapes (LineStrings) and returns the nearest point on each shape.

    Args:
        lon (float): Longitude
        lat (float): Latitude
        radius (int): Search radius in meters (default 1600 = ~20 min walking)
        dbname, user, password, host, port: Postgres connection params

    Returns:
        List of dicts: [{shape_id, feed_id, dist_meters, closest_point}]
    """

    query = """
    WITH my_point AS (
      SELECT ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography AS geom
    ),
    ranked AS (
      SELECT s.shape_id,
             s.feed_id,
             ST_ClosestPoint(s.geom, p.geom::geometry) AS closest_point,
             ST_Distance(s.geom::geography, p.geom) AS dist_meters
      FROM gtfs_shapes s, my_point p
      WHERE ST_DWithin(s.geom::geography, p.geom, %s)
    )
    SELECT shape_id,
           feed_id,
           ST_AsGeoJSON(closest_point) AS closest_point,
           dist_meters
    FROM ranked
    ORDER BY dist_meters;
    """

    conn = psycopg2.connect(
        dbname=dbname, user=user, password=password, host=host, port=port
    )
    cur = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)

    cur.execute(query, (lon, lat, radius))
    rows = cur.fetchall()

    cur.close()
    conn.close()

    return [
        {
            "shape_id": row["shape_id"],
            "feed_id": row["feed_id"],
            "closest_point": row["closest_point"],  # GeoJSON string
            "dist_meters": float(row["dist_meters"])
        }
        for row in rows
    ]


In [4]:
routes = get_near_routes(29.96139328537071, 31.22968895248673)

for r in routes:
    print(r["shape_id"], r["dist_meters"], r["closest_point"])


abuqir 521.29826406 {"type":"Point","coordinates":[29.957655872,31.233196371]}


In [8]:
import psycopg2
import psycopg2.extras

def get_route_path(lon, lat, shape_id,
                   dbname="osm", user="postgres",
                   password="67500", host="localhost", port=5433):
    """
    Compute walking route from a point (lon, lat) to the closest point on a GTFS route (shape_id).
    Uses OSM ways + pgRouting for shortest path.

    Returns:
        dict with {shape_id, closest_point, walk_geom, walk_distance_m}
    """

    query = """
    WITH input AS (
      SELECT ST_SetSRID(ST_Point(%s, %s), 4326) AS geom
    ),
    start_node AS (
      SELECT id
      FROM ways_vertices_pgr
      ORDER BY the_geom <-> (SELECT geom FROM input)
      LIMIT 1
    ),
    target_node AS (
      SELECT v.id, cp AS closest_point
      FROM gtfs_shapes s
      CROSS JOIN LATERAL ST_ClosestPoint(s.geom, (SELECT geom FROM input)) cp
      JOIN ways_vertices_pgr v
        ON v.id = (
          SELECT id FROM ways_vertices_pgr
          ORDER BY the_geom <-> cp
          LIMIT 1
        )
      WHERE s.shape_id = %s
      ORDER BY s.geom <-> (SELECT geom FROM input)
      LIMIT 1
    ),
    walk_path AS (
      SELECT e.the_geom, e.length_m
      FROM pgr_dijkstra(
        'SELECT gid AS id, source, target, length_m AS cost FROM ways',
        (SELECT id FROM start_node),
        (SELECT id FROM target_node),
        false
      ) dj
      JOIN ways e ON dj.edge = e.gid
    )
    SELECT %s::text AS shape_id,
          ST_AsGeoJSON(MIN(t.closest_point)) AS closest_point,
          ST_AsGeoJSON(ST_LineMerge(ST_Union(w.the_geom))) AS walk_geom,
          SUM(w.length_m) AS walk_distance_m
    FROM walk_path w, target_node t;
    """

    conn = psycopg2.connect(
        dbname=dbname, user=user, password=password, host=host, port=port
    )
    cur = conn.cursor(cursor_factory=psycopg2.extras.DictCursor)

    cur.execute(query, (lon, lat, shape_id, shape_id))
    row = cur.fetchone()

    cur.close()
    conn.close()

    if row:
        return {
            "shape_id": row["shape_id"],
            "closest_point": row["closest_point"],   # GeoJSON
            "walk_geom": row["walk_geom"],           # GeoJSON linestring
            "walk_distance_m": float(row["walk_distance_m"]) if row["walk_distance_m"] else None
        }
    else:
        return None


In [9]:
routes = get_near_routes(29.96139328537071, 31.22968895248673)

for r in routes:
    print(r["shape_id"], r["dist_meters"], r["closest_point"])
    route_path = get_route_path(29.96139328537071, 31.22968895248673, r["shape_id"])
    print(route_path)

abuqir 521.29826406 {"type":"Point","coordinates":[29.957655872,31.233196371]}
{'shape_id': 'abuqir', 'closest_point': '{"type":"Point","coordinates":[29.957655872,31.233196371]}', 'walk_geom': '{"type":"LineString","coordinates":[[29.9578107,31.2333546],[29.9574275,31.2329471],[29.9572487,31.2327568],[29.9572227,31.2327292],[29.9572887,31.2326796],[29.9573467,31.2326515],[29.9574044,31.2326375],[29.9574741,31.2326289],[29.9575206,31.2326275],[29.9575462,31.2326088],[29.9575982,31.2325468],[29.9576213,31.2325074],[29.9576713,31.2324343],[29.9583613,31.2327508],[29.9587212,31.2320943],[29.9588946,31.2317797],[29.9592379,31.2319415],[29.959764,31.2313436],[29.960028,31.2310384],[29.9601713,31.2308763],[29.9602456,31.2307911],[29.9604975,31.230502],[29.9608083,31.2301388],[29.9608308,31.2301125],[29.9609652,31.2299555],[29.9611636,31.2297165],[29.9614619,31.2298856]]}', 'walk_distance_m': 719.032803979558}


In [ ]:
# display the walk_geom on map
import folium
m = folium.Map(location=[31.22968895248673, 29.96139328537071], zoom_start=15)
folium.Marker(location=[31.22968895248673, 29.96139328537071], popup="Start").add_to(m)
if route_path and route_path["walk_geom"]:
    folium.GeoJson(route_path["walk_geom"], name="Walk Route").add_to(m)

# save to html
m.save("walk_route.html")

In [13]:
ends = get_near_routes(29.94194179397711, 31.20775934404925)
for e in ends:
    print(e["shape_id"], e["dist_meters"], e["closest_point"])

sidi_ezba 56.14346731 {"type":"Point","coordinates":[29.941697658,31.208224523]}
ezba_sidi 90.88504983 {"type":"Point","coordinates":[29.941022,31.207976]}
abuqir 896.11606435 {"type":"Point","coordinates":[29.935929796,31.21409386]}


In [6]:
import folium
import psycopg2
import json

def plot_routes(shape_ids, dbname="test", user="postgres", password="67500", host="localhost", port=5433):
    conn = psycopg2.connect(dbname=dbname, user=user, password=password, host=host, port=port)
    cur = conn.cursor()

    # Get route lines
    cur.execute("""
        SELECT shape_id, ST_AsGeoJSON(geom)
        FROM gtfs_shapes
        WHERE shape_id = ANY(%s)
    """, (shape_ids,))
    routes = cur.fetchall()

    # Get route points
    cur.execute("""
        SELECT shape_id, shape_pt_sequence,
               ST_AsGeoJSON(ST_SetSRID(ST_MakePoint(shape_pt_lon, shape_pt_lat),4326))
        FROM gtfs_shapes_raw
        WHERE shape_id = ANY(%s)
        ORDER BY shape_id, shape_pt_sequence
    """, (shape_ids,))
    points = cur.fetchall()

    cur.close()
    conn.close()

    # Make map centered on first route
    m = folium.Map(location=[31.23, 29.96], zoom_start=13)

    # Add routes
    for shape_id, geom in routes:
        coords = json.loads(geom)["coordinates"]
        coords_latlon = [(y, x) for x, y in coords]  # swap lon/lat
        folium.PolyLine(coords_latlon, weight=4, tooltip=shape_id).add_to(m)

    # Add points
    for shape_id, seq, geom in points:
        coords = json.loads(geom)["coordinates"]
        folium.CircleMarker(
            location=[coords[1], coords[0]],
            radius=3,
            color="red",
            fill=True,
            tooltip=f"{shape_id} seq {seq}"
        ).add_to(m)

    return m

# Example
m = plot_routes(["abuqir", "ezba_sidi", "sidi_ezba"])
m.save("routes_map.html")


In [ ]:
'''
pathways
-> abuqir (r1) -> [sidi_ezba (r2) : 89 → 1] 
-> sidi_ezba (r2) -> [ezba_sidi (r3) : 55 → 1]
-> ezba_sidi (r3) -> [abuqir (r1) : 26 → 89]

now the graph is:
r1 -> r2
r2 -> r3
r3 -> r1
'''

'\npathways\n-> abuqir (r1) -> [sidi_ezba (r2) : 89 → 1]\n-> sidi_ezba (r2) -> [ezba_sidi (r3) : 55 → 1]\n-> ezba_sidi (r3) -> [abuqir (r1) : 26 → 89]\n\nnow the graph is:\nr1 -> r2\nr2 -> r3\nr3 -> r1\n'

In [ ]:
# from previous queries ,
# target start  = r1
# target end = r1, r2, r3

In [14]:
# build the graph
graph = {
    "r1": ["r2"],
    "r2": ["r3"],
    "r3": ["r1"]
}

# for k levels, do BFS to find all reachable nodes
# k should be from 1 to n
# in each round , i check the routes in current level if one of them is the target then add it to results else expand this routes's child to the next level 
# when i move to a node, it should have all the nodes i visited before, and check that i didn't vist the same node again in the same path
def bfs_paths(graph, start, goals, k):
    results = [] # complete route results will be stored here
    queue = [(start, [start])] # a queue for the bfs, with the current node and it's accumulated path
    if start in goals: # if current route is in the target routes add it to the results
        results.append([start])
    while queue:
        (vertex, path) = queue.pop(0) # get the current route and it's path
        if len(path) - 1 < k:  # only expand if we haven't reached k levels
            for next in graph.get(vertex, []): # loop through routes reachable from the current node 
                if next not in path:  # avoid cycles if the node was already visited in the path
                    new_path = path + [next] # append to the current path
                    if next in goals: # if the new node is in the target list, append it to the results
                        results.append(new_path)
                    queue.append((next, new_path)) # add the new child to the queue
    return results

# Example usage
start = "r1"
goals = {"r1", "r2", "r3"}
k = 3
paths = bfs_paths(graph, start, goals, k)
for p in paths:
    print(" -> ".join(p))


r1
r1 -> r2
r1 -> r2 -> r3


In [ ]:
# TODO : calculate time taken in the transits from start stops to end stops
# TODO : calculate walking distances to start and and and between transfers and it's expected walking time (pgrouting)
# TODO : calculate total expected cost (clustering, cost prediction)
# TODO : create a list with tubles of (transfers, walking distance, total time, cost)
# TODO : filter routes that has extra transfers and doesn't optimize any other factor
# TODO : print full routes data and display on map ex : (walk to stop x take r1 depart at stop y, walk to stop z take r2 depart at w and walk to target)